In [93]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from pydantic import BaseModel,Field
from typing import Literal,TypedDict
from dotenv import load_dotenv
import os
from langchain_core.messages import SystemMessage, HumanMessage

In [94]:
load_dotenv(dotenv_path=".env", override=True)

True

In [95]:
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [96]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",  
    api_key=GROQ_API_KEY  
)

In [97]:
generator_llm=model
evaluator_llm=model
optimizer_llm=model

In [98]:
class poststate(TypedDict):
    title: str
    tweet:str
    evaluation : Literal['approved','not_approved']
    feedback:str
    

In [99]:
graph=StateGraph(poststate)

In [100]:
def generate(state:poststate)->poststate:
    
    title=state['title']
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{title}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]
    response=generator_llm.invoke(messages)
    return {'tweet':response.content}

In [ ]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "not_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [102]:
structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [103]:
def evaluate(state:poststate):
    
    messages = [
        SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet formaton ."),
        HumanMessage(content=f"""
        Evaluate the following tweet:

        Tweet: "{state['tweet']}"

        Use the criteria below to evaluate the tweet:

        1. Originality – Is this fresh, or have you seen it a hundred times before?  
        2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
        3. Punchiness – Is it short, sharp, and scroll-stopping?  
        4. Virality Potential – Would people retweet or share it?  
        5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

        Auto-reject if:
        - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
        - It exceeds 280 characters
        - It reads like a traditional setup-punchline joke
        - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

        ### Respond ONLY in structured format:
        - evaluation: "approved" or "needs_improvement"  
        - feedback: One paragraph explaining the strengths and weaknesses 
        """)
    ]
    response=structured_evaluator_llm.invoke(messages)
    return {'evaluation',response.evaluation,'feedback',response.feedback}

    

In [104]:
def check_condition(state:poststate):
    if state['evaluation']=='approved':
        return 'approved'
    else:
        return 'not_approved'

In [105]:

def optimized(state:poststate):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]

    response = optimizer_llm.invoke(messages).content
    

    return {'tweet': response}


In [106]:
graph.add_node('generate',generate)
graph.add_node('evaluate',evaluate)
graph.add_node('optimized',optimized)

In [107]:
graph.add_edge(START,'generate')
graph.add_edge('generate','evaluate')
graph.add_conditional_edges('evaluate',check_condition,{'approved':END,'not_approved':'optimized'})
graph.add_edge('optimized','evaluate')


In [108]:
workflow=graph.compile()
result=workflow.invoke({'title':'AI taking over the world'})

InvalidUpdateError: Expected dict, got {'approved', "This tweet is a great example of a well-crafted joke that meets most of the evaluation criteria. It has a good level of originality, as the comparison between a robot boss and a human boss is a fresh take on the AI takeover trope. The humor is also well-executed, as the relatable frustration with annoying bosses makes the joke more endearing and laughable. The tweet is punchy, short, and scroll-stopping, making it likely to grab the reader's attention. Additionally, the tweet has a high virality potential, as people can easily retweet or share it due to its witty humor and relatability. The format is also well-done, as it's under 280 characters and doesn't follow a traditional setup-punchline joke structure, making it a strong tweet overall.", 'feedback', 'evaluation'}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE

In [ ]:
result

{'title': 'AI taking over the world',
 'tweet': AIMessage(content='"AI taking over the world? Great, just what I need, a robot boss breathing down my neck, judging my Netflix binge hours"', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 119, 'total_tokens': 148, 'completion_time': 0.116473851, 'completion_tokens_details': None, 'prompt_time': 0.005388712, 'prompt_tokens_details': None, 'queue_time': 0.237626925, 'total_time': 0.121862563}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019de275-de2b-7701-adc7-d965a1d1c418-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 29, 'total_tokens': 148})}